In [40]:
# Sentiment Analysis for Stock and Twitter Data
# This notebook performs sentiment analysis on tweets and correlates with stock price movements

import pandas as pd
from datetime import timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import os

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

In [41]:
# Variable to do for the given ticker
ticker = "AAPL"

In [42]:
# Load the data files
print("Loading stock market data...")
stock_data = pd.read_csv(f'../data/{ticker}/{ticker}_market_clean.csv')
print(f"Date range: {stock_data['date'].min()} to {stock_data['date'].max()}")

print("\nLoading tweet data...")
tweet_data = pd.read_csv(f'../data/{ticker}/{ticker}_tweets_clean.csv')
print(f"Date range: {tweet_data['created_at'].min()} to {tweet_data['created_at'].max()}")

Loading stock market data...
Date range: 2020-10-01 to 2025-09-30

Loading tweet data...
Date range: 2020-10-01 to 2025-09-30


In [43]:
# Data preprocessing and exploration
print("Data preprocessing...")

# Convert date columns to datetime
stock_data['date'] = pd.to_datetime(stock_data['date'])
tweet_data['created_at'] = pd.to_datetime(tweet_data['created_at'])

# Extract date from tweet timestamps (keep only the date part)
tweet_data['date'] = tweet_data['created_at'].dt.date

# Check unique dates in each dataset
print(f"\nUnique dates in stock data: {len(stock_data['date'].dt.date.unique())}")
print(f"Unique dates in tweet data: {len(tweet_data['date'].unique())}")

Data preprocessing...

Unique dates in stock data: 1255
Unique dates in tweet data: 1764


In [44]:
# VADER sentiment analysis
def analyze_sentiment(text):
    """
    Analyze sentiment using VADER (Valence Aware Dictionary and sEntiment Reasoner)
    Returns polarity score (-1 to 1, where -1 is very negative, 1 is very positive)
    VADER is specifically designed for social media text and handles emojis, slang, etc.
    """
    try:
        # Get sentiment scores from VADER
        scores = analyzer.polarity_scores(str(text))
        
        # Return the compound score which is the overall sentiment
        # Compound score ranges from -1 (most negative) to +1 (most positive)
        return scores['compound']
    except:
        return 0.0  # Return neutral sentiment for problematic text

# Apply sentiment analysis to tweets
print("Performing sentiment analysis on tweets...")
tweet_data['sentiment'] = tweet_data['text'].apply(analyze_sentiment)

# Show distribution of sentiment scores
print(f"\nSentiment distribution:")
print(tweet_data['sentiment'].describe())


Performing sentiment analysis on tweets...

Sentiment distribution:
count    38553.000000
mean         0.157659
std          0.413466
min         -0.994500
25%          0.000000
50%          0.000000
75%          0.469600
max          0.999300
Name: sentiment, dtype: float64


In [45]:
# Group tweets by date and calculate average sentiment
print("Grouping tweets by date and calculating average sentiment...")

# Group by date and calculate average sentiment
daily_sentiment = tweet_data.groupby('date')['sentiment'].agg(['mean', 'count']).reset_index()
daily_sentiment.columns = ['date', 'average_sentiment', 'tweet_count']

# Show first few rows
print("\nDaily sentiment preview:")
print(daily_sentiment.head())

# Show tweet count distribution
print(f"\nTweet count per day statistics:")
print(daily_sentiment['tweet_count'].describe())


Grouping tweets by date and calculating average sentiment...

Daily sentiment preview:
         date  average_sentiment  tweet_count
0  2020-10-01           0.264496           25
1  2020-10-02           0.105988           25
2  2020-10-03           0.230556           25
3  2020-10-04           0.247916           25
4  2020-10-05           0.237484           25

Tweet count per day statistics:
count    1764.000000
mean       21.855442
std         7.091899
min         1.000000
25%        25.000000
50%        25.000000
75%        25.000000
max        25.000000
Name: tweet_count, dtype: float64


In [46]:
# Handle weekend grouping - group Saturday and Sunday with the previous Friday

# Convert date to datetime for easier manipulation
daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])

# Create a mapping for weekend grouping
def get_grouping_date(date):
    """
    For weekends (Saturday=5, Sunday=6), group with the previous Friday
    For weekdays, keep the original date
    """
    if date.weekday() == 5:  # Saturday
        return date - timedelta(days=1)  # Group with Friday
    elif date.weekday() == 6:  # Sunday
        return date - timedelta(days=2)  # Group with Friday
    else:
        return date  # Keep original date for weekdays

# Apply weekend grouping
daily_sentiment['grouping_date'] = daily_sentiment['date'].apply(get_grouping_date)

# Group by the new grouping date and recalculate sentiment
grouped_sentiment = daily_sentiment.groupby('grouping_date').agg({
    'average_sentiment': 'mean',  # Average of averages
    'tweet_count': 'sum'  # Sum tweet counts
}).reset_index()

grouped_sentiment.columns = ['date', 'average_sentiment', 'tweet_count']

print(f"After weekend grouping:")
print(f"Grouped sentiment data shape: {grouped_sentiment.shape}")

# Show some examples of grouped data
print(f"\nGrouped sentiment preview:")
print(grouped_sentiment.head(5))


After weekend grouping:
Grouped sentiment data shape: (1281, 3)

Grouped sentiment preview:
        date  average_sentiment  tweet_count
0 2020-10-01           0.264496           25
1 2020-10-02           0.194820           75
2 2020-10-05           0.237484           25
3 2020-10-06           0.103212           25
4 2020-10-07           0.135392           25


In [47]:
# Calculate price differences (next day open - current day close)
print("Calculating price differences...")

# Sort stock data by date to ensure proper order
stock_data_sorted = stock_data.sort_values('date').reset_index(drop=True)

# Calculate the difference between next day's open and current day's close
stock_data_sorted['next_day_open'] = stock_data_sorted['open'].shift(-1)
stock_data_sorted['price_difference'] = stock_data_sorted['next_day_open'] - stock_data_sorted['close']

# Remove the last row since it won't have a next day
stock_data_sorted = stock_data_sorted[:-1]

print(f"Stock data with price differences shape: {stock_data_sorted.shape}")
print(f"Price difference range: {stock_data_sorted['price_difference'].min():.3f} to {stock_data_sorted['price_difference'].max():.3f}")
print(f"Mean price difference: {stock_data_sorted['price_difference'].mean():.3f}")

# Show first few rows with price differences
print(f"\nStock data with price differences preview:")
print(stock_data_sorted[['date', 'open', 'close', 'next_day_open', 'price_difference']].head(10))


Calculating price differences...
Stock data with price differences shape: (1254, 5)
Price difference range: -20.650 to 13.523
Mean price difference: -0.062

Stock data with price differences preview:
        date        open       close  next_day_open  price_difference
0 2020-10-01  114.430523  113.603714     109.810103         -3.793611
1 2020-10-02  109.810103  109.936554     110.802274          0.865720
2 2020-10-05  110.802274  113.321609     112.543443         -0.778167
3 2020-10-06  112.543443  110.072746     111.492935          1.420188
4 2020-10-07  111.492935  111.940384     113.078452          1.138068
5 2020-10-08  113.078452  111.833374     112.134921          0.301547
6 2020-10-09  112.134921  113.778816     116.784475          3.005658
7 2020-10-12  116.784475  121.006073     121.852359          0.846286
8 2020-10-13  121.852359  117.796127     117.698863         -0.097264
9 2020-10-14  117.698863  117.883682     115.481064         -2.402618


In [48]:
# Merge sentiment data with stock data
print("Merging sentiment and stock data...")

# Convert date columns to the same format for merging
grouped_sentiment['date'] = pd.to_datetime(grouped_sentiment['date']).dt.date
stock_data_sorted['date'] = stock_data_sorted['date'].dt.date

# Merge the datasets
final_dataset = pd.merge(
    grouped_sentiment, 
    stock_data_sorted[['date', 'price_difference']], 
    on='date', 
    how='inner'  # Only keep dates that exist in both datasets
)

# Show the final dataset structure
print(f"\nFinal dataset preview:")
print(final_dataset.head(10))

# Show statistics for the final dataset
print(f"\nFinal dataset statistics:")
print(final_dataset.describe())


Merging sentiment and stock data...

Final dataset preview:
         date  average_sentiment  tweet_count  price_difference
0  2020-10-01           0.264496           25         -3.793611
1  2020-10-02           0.194820           75          0.865720
2  2020-10-05           0.237484           25         -0.778167
3  2020-10-06           0.103212           25          1.420188
4  2020-10-07           0.135392           25          1.138068
5  2020-10-08           0.278152           25          0.301547
6  2020-10-09           0.253848           75          3.005658
7  2020-10-12           0.281336           25          0.846286
8  2020-10-13           0.234436           25         -0.097264
9  2020-10-14           0.431412           25         -2.402618

Final dataset statistics:
       average_sentiment  tweet_count  price_difference
count        1235.000000  1235.000000       1235.000000
mean            0.160609    30.218623         -0.063815
std             0.120556    19.594656    

In [49]:
# Update the merge to include open and close prices with correct alignment
print("Updating merge to include open and close prices with correct alignment...")

# Create properly aligned stock data:
# - close: current day's close price
# - next_day_open: next day's open price (for price_difference calculation)
# - We need to shift the open prices to represent next day's open

# Create a copy of stock data with shifted open prices
stock_aligned = stock_data_sorted.copy()
stock_aligned['next_day_open'] = stock_data_sorted['open'].shift(-1)

# Re-merge the datasets with properly aligned prices
final_dataset = pd.merge(
    grouped_sentiment, 
    stock_aligned[['date', 'close', 'next_day_open', 'price_difference']], 
    on='date', 
    how='inner'  # Only keep dates that exist in both datasets
)

# Rename columns for clarity
final_dataset = final_dataset.rename(columns={
    'close': 'close',
    'next_day_open': 'open'  # This is actually the next day's open
})

print(f"Updated dataset shape: {final_dataset.shape}")
print(f"Updated dataset columns: {list(final_dataset.columns)}")
print(f"\nSample of aligned data:")
print(final_dataset[['date', 'open', 'close', 'price_difference']].head())


Updating merge to include open and close prices with correct alignment...
Updated dataset shape: (1235, 6)
Updated dataset columns: ['date', 'average_sentiment', 'tweet_count', 'close', 'open', 'price_difference']

Sample of aligned data:
         date        open       close  price_difference
0  2020-10-01  109.810103  113.603714         -3.793611
1  2020-10-02  110.802274  109.936554          0.865720
2  2020-10-05  112.543443  113.321609         -0.778167
3  2020-10-06  111.492935  110.072746          1.420188
4  2020-10-07  113.078452  111.940384          1.138068


In [50]:
# Create the final dataset with the required columns
print("Creating final sentiment dataset...")

# Select only the required columns: date, average_sentiment, price_difference
sentiment_final = final_dataset[['date', 'average_sentiment', 'price_difference']].copy()

# Show the final dataset
print(f"\nFinal sentiment dataset preview:")
print(sentiment_final.head(10))

# Show summary statistics
print(f"\nSummary statistics:")
print(sentiment_final.describe())


Creating final sentiment dataset...

Final sentiment dataset preview:
         date  average_sentiment  price_difference
0  2020-10-01           0.264496         -3.793611
1  2020-10-02           0.194820          0.865720
2  2020-10-05           0.237484         -0.778167
3  2020-10-06           0.103212          1.420188
4  2020-10-07           0.135392          1.138068
5  2020-10-08           0.278152          0.301547
6  2020-10-09           0.253848          3.005658
7  2020-10-12           0.281336          0.846286
8  2020-10-13           0.234436         -0.097264
9  2020-10-14           0.431412         -2.402618

Summary statistics:
       average_sentiment  price_difference
count        1235.000000       1235.000000
mean            0.160609         -0.063815
std             0.120556          1.967003
min            -0.367550        -20.649896
25%             0.088546         -0.785486
50%             0.155621          0.029696
75%             0.226491          0.757871
max 

In [51]:
# Update the final dataset to include open and close prices
print("Updating final dataset to include open and close prices...")

# Select the required columns: date, average_sentiment, open, close, price_difference
sentiment_final = final_dataset[['date', 'average_sentiment', 'open', 'close', 'price_difference']].copy()

print(f"Updated final dataset shape: {sentiment_final.shape}")
print(f"Updated final dataset columns: {list(sentiment_final.columns)}")


Updating final dataset to include open and close prices...
Updated final dataset shape: (1235, 5)
Updated final dataset columns: ['date', 'average_sentiment', 'close', 'open', 'price_difference']

Dropping NA/empty rows...
Rows before dropping NA: 1235
Rows after dropping NA: 1234

Verification - Sample of final dataset:
         date       close        open  price_difference
0  2020-10-01  113.603714  109.810103         -3.793611
1  2020-10-02  109.936554  110.802274          0.865720
2  2020-10-05  113.321609  112.543443         -0.778167
3  2020-10-06  110.072746  111.492935          1.420188
4  2020-10-07  111.940384  113.078452          1.138068

Note: 'close' column represents the CURRENT day's close price
      'open' column now represents the NEXT day's open price
      'price_difference' = next_day_open - current_day_close


In [52]:
# Save the final dataset
print("Saving the final sentiment dataset...")

# Save to the specified location
output_path = f'../data/{ticker}/{ticker}_sentiment.csv'
sentiment_final.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")

# Verify the saved file
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f"File saved successfully! Size: {file_size} bytes")
    
    # Read back a few rows to verify
    verification = pd.read_csv(output_path)
    print(f"\nVerification - first 5 rows of saved file:")
    print(verification.head())
    print(f"\nColumns in saved file: {list(verification.columns)}")
else:
    print("Error: File was not saved successfully!")


Saving the final sentiment dataset...
Dataset saved to: ../data/AAPL/AAPL_sentiment.csv
File saved successfully! Size: 100950 bytes

Verification - first 5 rows of saved file:
         date  average_sentiment       close        open  price_difference
0  2020-10-01           0.264496  113.603714  109.810103         -3.793611
1  2020-10-02           0.194820  109.936554  110.802274          0.865720
2  2020-10-05           0.237484  113.321609  112.543443         -0.778167
3  2020-10-06           0.103212  110.072746  111.492935          1.420188
4  2020-10-07           0.135392  111.940384  113.078452          1.138068

Columns in saved file: ['date', 'average_sentiment', 'close', 'open', 'price_difference']
